# Phase 1c: perception diagnostics for CheXagent-2-3b

Run 3 found two perception faults in the same model: a constant **"Right"** on explicit "which side" questions
(26 of 29, whatever the image) and **"no"** to heart-enlargement questions whose answer was "yes". These three cheap
tests ask what could explain them. No training and no retrieval; same model and settings as the other notebooks.

| Test | Question | How |
|---|---|---|
| **A. Flip** | Does the model read left/right from the image at all? | Mirror each image. If it reads laterality its answers should swap; a constant answer will not. |
| **B. Prompt variants** | Is the "Right" habit a wording effect? | Six wordings on 60 side questions, including a forced choice, describe-first, and stating the left/right convention. |
| **C. Heart size** | Can the model ever say "enlarged"? | 47 heart questions (22 with answer "yes") with four wordings, including "normal or enlarged?". |

**What each outcome would mean was written down before running** in `docs/phase1c_diagnostics.md`. Read it first.

**Runtime:** free Colab **T4**. Full run is about 700 short generations (expect roughly 30-60 minutes). Set `QUICK = True`
in the build cell for a ten-minute smoke test first. Progress is saved to `diagnostics_checkpoint.json`, so a dropped session
can resume: re-run the cells and finished answers are skipped.


## 1. Setup

Identical to the other notebooks. If Colab asks you to restart the runtime after this cell, do so and continue from the next cell.

In [ ]:
#@title Install dependencies (torch is preinstalled on Colab)
!pip install -q "transformers==4.46.3" accelerate einops sentencepiece protobuf pyarrow albumentations huggingface_hub
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")


## 2. Build the test items

The cells below are the repo's own modules, inlined unchanged. Order matters.

In [ ]:
"""Heuristic question-type categorization.

The literature review (CheXagent in particular) found that CXR VLMs score
~90-99% on coarse perception questions (presence/absence, modality, view,
organ identification) but drop to ~77-79% on fine-grained laterality and
severity questions ("which side", "mild vs. severe"). To test whether our
baseline shows the same weak spot, every question in the baseline test set
is tagged with a category up front so results can be broken down by category
rather than reported as one aggregate accuracy number.

This is a deliberately simple keyword heuristic, not a trained classifier.
It will mis-tag some edge cases (e.g. "what is to the left of the heart" is
positional, not a laterality-of-finding question) -- that is an accepted,
documented limitation, not a bug to silently paper over.
"""


import re

LATERALITY_PATTERNS = [
    r"\bleft\b",
    r"\bright\b",
    r"\bbilateral\b",
    r"\bwhich side\b",
    r"\blaterality\b",
    r"\bunilateral\b",
]

SEVERITY_PATTERNS = [
    r"\bmild\b",
    r"\bmoderate\b",
    r"\bsevere\b",
    r"\benlarge(d|ment)?\b",
    r"\bdegree\b",
    r"\bextent\b",
    r"\bhow (much|severe|large|big)\b",
    r"\bsize of\b",
    r"\bworsen(ed|ing)?\b",
    r"\bimprov(ed|ing|ement)\b",
    r"\bsmall\b",
    r"\blarge\b",
]

KNOWLEDGE_PATTERNS = [
    r"\bsymptoms?\b",
    r"\bhow to (treat|prevent|cure)\b",
    r"\b(treatment|therapy|prevention)\b",
    r"\bfunction of\b",
    r"\bcause[sd]?\b",
    r"\bhow do you (determine|diagnose|assess|measure)\b",
    # Added with the knowledge test set: 'why ...?' and 'how can you tell ...?'
    # are answered from anatomy/physiology knowledge. Run 2's saved tags were
    # produced before these two were added.
    r"\bwhy\b",
    r"\bhow can you tell\b",
]

# Questions that are templates about the imaging procedure, not about the
# patient's findings (e.g. SLAKE's "How much damage will be caused to the body
# taking such an image?"). They match the severity keywords ("how much") but are
# not clinical severity, so the data pipeline drops them.
NON_CLINICAL_PATTERNS = [
    r"\bdamage\b.*\btaking such an image\b",
    r"\bradiation\b",
]

CATEGORY_LATERALITY = "laterality"
CATEGORY_SEVERITY = "severity"
CATEGORY_GENERAL = "general"

ALL_CATEGORIES = (CATEGORY_LATERALITY, CATEGORY_SEVERITY, CATEGORY_GENERAL)


def _matches_any(text: str, patterns: list[str]) -> bool:
    return any(re.search(p, text) for p in patterns)


def categorize_question(question: str) -> str:
    """Tag a question as 'laterality', 'severity', or 'general'.

    Laterality is checked before severity because a question can contain
    both kinds of words ("is the left-sided effusion severe?") -- in our
    taxonomy laterality takes priority since it is the harder, more
    specifically-flagged failure mode in CheXagent's own reported numbers.
    """
    q = question.lower().strip()
    if _matches_any(q, LATERALITY_PATTERNS):
        return CATEGORY_LATERALITY
    if _matches_any(q, SEVERITY_PATTERNS):
        return CATEGORY_SEVERITY
    return CATEGORY_GENERAL


def is_knowledge_question(question: str) -> bool:
    """True for questions answered mostly from medical knowledge (symptoms,
    treatment, prevention, how a finding is measured) rather than from looking
    at the image. These are what a retrieval layer can plausibly help with;
    perception questions (is X present, which side) are not.

    Independent of `categorize_question`: a question can be both a laterality
    question and a knowledge question ("How to treat the disease on the lower
    left of lung?")."""
    q = question.lower().strip()
    return _matches_any(q, KNOWLEDGE_PATTERNS)


_SUBTYPE_RULES = [
    ("symptoms", r"\bsymptoms?\b"),
    ("treatment", r"\b(how to treat|treatment|therapy)\b"),
    ("prevention", r"\b(prevent|prevention)\b"),
    ("cause", r"\bcause[sd]?\b"),
    ("function", r"\bfunction of\b"),
    ("explanation", r"\b(why|how can you tell|how do you (determine|diagnose|assess|measure))\b"),
]


def knowledge_subtype(question: str) -> str:
    """Which kind of knowledge question this is (symptoms / treatment /
    prevention / cause / function / explanation / other). Reported separately
    because the SLAKE knowledge questions come from a few fixed templates and
    a model can score well on one template and not another."""
    q = question.lower().strip()
    for name, pattern in _SUBTYPE_RULES:
        if re.search(pattern, q):
            return name
    return "other"


def is_non_clinical_question(question: str) -> bool:
    q = question.lower().strip()
    return _matches_any(q, NON_CLINICAL_PATTERNS)


def answer_format(gold_answer: str) -> str:
    """'yes/no' for closed questions, 'open' otherwise. Reported separately
    because a model that mostly answers 'no' looks decent on yes/no questions
    when most true answers are 'no'."""
    g = gold_answer.lower().strip().rstrip(".")
    return "yes/no" if g in ("yes", "no") else "open"


In [ ]:
"""Build a small, chest-X-ray-only VQA test set for Phase 1 baseline testing.

Important caveat (documented here, not hidden): CheXagent's own training
corpus (CheXinstruct) explicitly rolls in VQA-RAD, SLAKE, and Open-I/IU X-ray
among its 32 source datasets. So evaluating CheXagent on these test splits is
*not* a clean held-out/zero-shot measurement -- it is close to in-distribution.
That is simply the state of public CXR VQA benchmarks today (virtually every
other paper in our review reuses the same splits for the same reason).

The point of Phase 1 is not to claim a novel generalization number; it is to
check whether our own baseline run reproduces the *specific, already-known*
weak spot from CheXagent's paper (strong on coarse/general questions, weaker
on laterality/severity) on data we control end-to-end, so Phase 2 (RAG) has a
concrete, measured gap to target rather than a hypothetical one.

Two sources are combined:
- SLAKE (test + validation): has structured `location` + `modality` metadata, so
  chest X-ray rows can be filtered precisely (no guessing from question text).
- VQA-RAD: has no organ metadata, so chest rows are selected with a keyword
  heuristic (noisier, clearly flagged below -- not a silent assumption).

Non-clinical template questions (e.g. SLAKE's radiation-damage question) are
dropped, repeated question text is capped per category, and every record gets a
stable `id` taken from the source dataset's own row id so later runs (RAG) can
be compared with this baseline row by row.
"""


import hashlib
import json
import random
import re
import time
import zipfile
from pathlib import Path

import requests


SLAKE_DATASET = "BoKelvin/SLAKE"
SLAKE_IMAGES_ZIP_URL = "https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip"
VQA_RAD_DATASET = "flaviagiammarino/vqa-rad"
DATASETS_SERVER_ROWS_URL = "https://datasets-server.huggingface.co/rows"

# Preferred data path: download each dataset's own files once (5 requests).
# The paging API (`datasets-server`, ~150 requests) rate-limits Colab with HTTP
# 429, so it is kept only as a fallback. Row ids are identical on both paths,
# checked against all 224 question ids of run 2.
HF_RESOLVE_URL = "https://huggingface.co/datasets/{repo}/resolve/main/{path}"
SLAKE_SPLIT_FILES = {"test": "test.json", "validation": "validation.json", "train": "train.json"}
VQA_RAD_PARQUET_FILES = {
    "test": "data/test-00000-of-00001-e5bc3d208bb4deeb.parquet",
    "train": "data/train-00000-of-00001-eb8844602202be60.parquet",
}
VQA_RAD_PARQUET_KEY_PREFIX = "vqarad-parquet"
RAW_CACHE_DIR = Path("raw_cache")

# SLAKE's `location` field uses these labels for chest X-ray images.
SLAKE_CHEST_LOCATIONS = {"Chest_heart", "Chest_lung", "Chest_mediastinal", "Lung"}

# VQA-RAD keyword heuristic: a question must mention a chest-related term and
# must NOT mention a clearly non-chest body region. This is approximate --
# some true negatives/positives will slip through -- which is why SLAKE
# (precise metadata) is the primary source and VQA-RAD is secondary/noisier.
VQA_RAD_CHEST_KEYWORDS = {
    "lung", "lungs", "chest", "pulmonary", "pleural", "pneumothorax",
    "cardiac", "heart", "mediastin", "rib", "costophrenic", "consolidation",
    "effusion", "atelectasis", "diaphragm", "thora", "cardiomegaly",
    "pneumonia", "hilar", "bronch",
}
# Keywords that must match as a whole word. "rib" is the one that matters: as a
# substring it matched "describe", "circumscribed", "distribution" and
# "tributaries", which let brain/abdominal "describe the lesion" questions into
# the chest set (found by looking at the images of the side set).
VQA_RAD_WHOLE_WORD_KEYWORDS = {"rib", "ribs"}
_WHOLE_WORD_RE = re.compile(r"\b(" + "|".join(sorted(VQA_RAD_WHOLE_WORD_KEYWORDS)) + r")\b")

VQA_RAD_EXCLUDE_KEYWORDS = {
    "brain", "skull", "head", "abdomen", "abdominal", "liver", "kidney",
    "pelvis", "pelvic", "spine", "spinal", "bowel", "intestine", "stomach",
    "bladder", "uterus", "ovary", "appendix", "cerebral", "cranial", "sinus",
    # Added after the pilot run surfaced non-chest items (e.g. "vascular
    # distribution of these infarcts"). Still a heuristic -- review the images.
    "infarct", "ventric", "sulci", "gyri", "white matter", "gray matter",
    "grey matter", "falx", "midline", "mass effect", "vascular distribution",
    "hepatic", "splenic", "spleen", "adrenal", "pancrea", "gallbladder",
    "renal", "psoas", "femur", "knee", "shoulder", "orbit",
}


def mentions_chest_term(question: str) -> bool:
    """True if the question text contains a chest keyword (substring match,
    except for short words listed in VQA_RAD_WHOLE_WORD_KEYWORDS)."""
    q = question.lower()
    if _WHOLE_WORD_RE.search(q):
        return True
    return any(k in q for k in VQA_RAD_CHEST_KEYWORDS if k not in VQA_RAD_WHOLE_WORD_KEYWORDS)


def _retry_wait(resp, attempt: int, backoff: float) -> float:
    """Seconds to wait before retrying. Honours Retry-After on HTTP 429 and
    otherwise backs off exponentially (capped at 60 s)."""
    if resp is not None and resp.status_code == 429:
        try:
            return min(120.0, max(float(resp.headers.get("Retry-After", 0)), backoff * 4 * (attempt + 1)))
        except ValueError:
            pass
    return min(60.0, backoff * (2 ** attempt))


def _download_file(url: str, dest: str | Path, retries: int = 8, backoff: float = 2.0) -> Path:
    """Download `url` to `dest` once (skipped if present), streaming, with
    retries on 429 / 5xx / connection errors. Written via a temp file so an
    interrupted download is never mistaken for a complete one."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    last_exc: Exception | None = None
    for attempt in range(retries):
        resp = None
        try:
            resp = requests.get(url, stream=True, timeout=180)
            if resp.status_code == 200:
                tmp = dest.with_suffix(dest.suffix + ".part")
                with open(tmp, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
                tmp.replace(dest)
                return dest
            if resp.status_code not in (429, 500, 502, 503, 504):
                resp.raise_for_status()
        except requests.RequestException as exc:
            last_exc = exc
            if resp is not None and resp.status_code not in (429, 500, 502, 503, 504):
                raise
        time.sleep(_retry_wait(resp, attempt, backoff))
    raise last_exc or RuntimeError(f"could not download {url}")


def _get_rows(dataset: str, split: str, offset: int, length: int = 100,
              retries: int = 8, backoff: float = 1.5) -> dict:
    last_exc = None
    for attempt in range(retries):
        try:
            resp = requests.get(
                DATASETS_SERVER_ROWS_URL,
                params={
                    "dataset": dataset, "config": "default", "split": split,
                    "offset": offset, "length": length,
                },
                timeout=30,
            )
            if resp.status_code == 200:
                return resp.json()
        except requests.RequestException as exc:
            last_exc = exc
            resp = None
        time.sleep(_retry_wait(resp, attempt, backoff))
    if last_exc:
        raise last_exc
    resp.raise_for_status()
    return {}


def _iter_all_rows(dataset: str, split: str, page_size: int = 100):
    offset = 0
    while True:
        data = _get_rows(dataset, split, offset, length=page_size)
        rows = data.get("rows", [])
        if not rows:
            return
        for row in rows:
            yield {**row["row"], "_row_idx": row["row_idx"]}
        offset += len(rows)
        if offset >= data.get("num_rows_total", 0):
            return


def _iter_slake_rows(split: str):
    """SLAKE rows from the dataset's own JSON file; paging API as fallback."""
    try:
        url = HF_RESOLVE_URL.format(repo=SLAKE_DATASET, path=SLAKE_SPLIT_FILES[split])
        path = _download_file(url, RAW_CACHE_DIR / f"slake_{split}.json")
        rows = json.loads(path.read_text())
    except Exception as exc:
        print(f"[data] SLAKE {split}: file download failed ({exc!r}); falling back to the paging API")
        yield from _iter_all_rows(SLAKE_DATASET, split)
        return
    yield from rows


def fetch_slake_chest_records(split: str = "test") -> list[dict]:
    """English chest-X-ray rows from SLAKE, filtered via structured metadata."""
    records = []
    for rr in _iter_slake_rows(split):
        if (
            rr.get("modality") == "X-Ray"
            and rr.get("location") in SLAKE_CHEST_LOCATIONS
            and rr.get("q_lang") == "en"
            and not is_non_clinical_question(rr["question"])
        ):
            records.append({
                "source_dataset": "SLAKE",
                "source_split": split,
                "question": rr["question"],
                "gold_answer": rr["answer"],
                "answer_type": rr.get("answer_type"),
                "uid": f"SLAKE_{rr['qid']}",
                "image_key": rr["img_name"],  # e.g. "xmlab102/source.jpg"
                "category": categorize_question(rr["question"]),
                "answer_format": answer_format(rr["answer"]),
                "is_knowledge": is_knowledge_question(rr["question"]),
            })
    return records


def _vqa_rad_parquet_path(split: str) -> Path:
    url = HF_RESOLVE_URL.format(repo=VQA_RAD_DATASET, path=VQA_RAD_PARQUET_FILES[split])
    return _download_file(url, RAW_CACHE_DIR / f"vqa_rad_{split}.parquet")


def _iter_vqa_rad_rows(split: str):
    """VQA-RAD rows from the dataset's parquet file (images are embedded, so
    there are no expiring signed URLs, and an image's content hash gives a real
    image identity); paging API as fallback if pyarrow or the download fails.

    Yields dicts with question, answer, _row_idx, _image_key, _image_id."""
    try:
        import pyarrow.parquet as pq
        table = pq.read_table(_vqa_rad_parquet_path(split))
        images, questions, answers = table.column("image"), table.column("question"), table.column("answer")
        rows = []
        for idx in range(table.num_rows):
            img_bytes = images[idx].as_py()["bytes"]
            rows.append({
                "question": questions[idx].as_py(), "answer": answers[idx].as_py(), "_row_idx": idx,
                "_image_key": f"{VQA_RAD_PARQUET_KEY_PREFIX}:{split}:{idx}",
                "_image_id": hashlib.md5(img_bytes).hexdigest()[:12],
            })
    except Exception as exc:
        print(f"[data] VQA-RAD {split}: parquet path failed ({exc!r}); falling back to the paging API")
        for rr in _iter_all_rows(VQA_RAD_DATASET, split):
            yield {"question": rr["question"], "answer": rr["answer"], "_row_idx": rr["_row_idx"],
                   "_image_key": rr["image"]["src"], "_image_id": None}
        return
    yield from rows


def fetch_vqa_rad_chest_records(splits: tuple[str, ...] = ("test", "train")) -> list[dict]:
    """Chest-ish rows from VQA-RAD, filtered via keyword heuristic (noisier)."""
    records = []
    for split in splits:
        for rr in _iter_vqa_rad_rows(split):
            q = rr["question"].lower()
            if (
                mentions_chest_term(q)
                and not any(k in q for k in VQA_RAD_EXCLUDE_KEYWORDS)
                and not is_non_clinical_question(rr["question"])
            ):
                records.append({
                    "source_dataset": "VQA-RAD",
                    "source_split": split,
                    "question": rr["question"],
                    "gold_answer": rr["answer"],
                    "answer_type": None,
                    "uid": f"VQA-RAD_{split}_{rr['_row_idx']}",
                    "image_key": rr["_image_key"],
                    "image_id": rr["_image_id"],  # content hash of the image (None on the fallback path)
                    "category": categorize_question(rr["question"]),
                    "answer_format": answer_format(rr["answer"]),
                    "is_knowledge": is_knowledge_question(rr["question"]),
                })
    return records


def stratified_sample(records: list[dict], n_per_category: int, seed: int = 0,
                      max_per_question: int = 4) -> list[dict]:
    """Sample up to `n_per_category` records per category.

    `max_per_question` caps how many rows with identical question text can
    be drawn (the pilot run had "is the heart enlarged?" ~12 times in 60 rows,
    which lets one question dominate a category). Deterministic for a given
    input order and seed.
    """
    rng = random.Random(seed)
    by_cat: dict[str, list[dict]] = {}
    for r in records:
        by_cat.setdefault(r["category"], []).append(r)
    sampled = []
    for cat in sorted(by_cat):
        shuffled = by_cat[cat][:]
        rng.shuffle(shuffled)
        per_question: dict[str, int] = {}
        taken = []
        for r in shuffled:
            key = r["question"].lower().strip()
            if per_question.get(key, 0) >= max_per_question:
                continue
            per_question[key] = per_question.get(key, 0) + 1
            taken.append(r)
            if len(taken) >= n_per_category:
                break
        sampled.extend(taken)
    rng.shuffle(sampled)
    return sampled


def pool_summary(records: list[dict]) -> dict[str, dict[str, int]]:
    """Counts per category / source / answer format, so it is visible how many
    questions of each kind actually exist (severity questions are rare in
    public CXR VQA sets -- that is itself a finding)."""
    summary: dict[str, dict[str, int]] = {"category": {}, "source": {}, "answer_format": {}}
    for r in records:
        for field, key in (("category", r["category"]), ("source", r["source_dataset"]),
                           ("answer_format", r["answer_format"])):
            summary[field][key] = summary[field].get(key, 0) + 1
    return summary


def download_slake_images(records: list[dict], dest_dir: str | Path,
                           zip_cache_path: str | Path | None = None) -> dict[str, str]:
    """Download SLAKE's imgs.zip once (cached) and extract only needed files."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    needed_keys = {r["image_key"] for r in records if r["source_dataset"] == "SLAKE"}
    if not needed_keys:
        return {}

    zip_cache_path = Path(zip_cache_path) if zip_cache_path else dest_dir / "_slake_imgs.zip"
    _download_file(SLAKE_IMAGES_ZIP_URL, zip_cache_path)

    local_paths = {}
    with zipfile.ZipFile(zip_cache_path) as zf:
        for key in needed_keys:
            member = f"imgs/{key}"
            out_path = dest_dir / key.replace("/", "__")
            if not out_path.exists():
                with zf.open(member) as src, open(out_path, "wb") as dst:
                    dst.write(src.read())
            local_paths[key] = str(out_path)
    return local_paths


def download_vqa_rad_images(records: list[dict], dest_dir: str | Path) -> dict[str, str]:
    """Write each needed VQA-RAD image to `dest_dir`. Records from the parquet
    path are extracted from the parquet file and named by image content hash, so
    questions about the same image share one file. Records from the paging
    fallback are fetched from their (expiring) signed URLs."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    local_paths: dict[str, str] = {}
    tables: dict[str, object] = {}
    for r in records:
        if r["source_dataset"] != "VQA-RAD":
            continue
        key = r["image_key"]
        if key in local_paths:
            continue
        if key.startswith(VQA_RAD_PARQUET_KEY_PREFIX + ":"):
            _, split, idx = key.split(":")
            out_path = dest_dir / f"vqarad_{r['image_id']}.jpg"
            if not out_path.exists():
                if split not in tables:
                    import pyarrow.parquet as pq
                    tables[split] = pq.read_table(_vqa_rad_parquet_path(split), columns=["image"])
                out_path.write_bytes(tables[split].column("image")[int(idx)].as_py()["bytes"])
        else:
            digest = hashlib.md5(key.encode()).hexdigest()[:12]
            out_path = dest_dir / f"vqarad_{digest}.jpg"
            if not out_path.exists():
                resp = requests.get(key, timeout=60)
                resp.raise_for_status()
                with open(out_path, "wb") as f:
                    f.write(resp.content)
        local_paths[key] = str(out_path)
    return local_paths


def build_baseline_test_set(images_dir: str | Path, n_per_category: int = 100,
                             seed: int = 0, max_per_question: int = 4,
                             verbose: bool = True) -> list[dict]:
    """End-to-end: fetch metadata, stratify-sample per category, download
    images, return records ready for model inference. Each record has an
    `image_path` and a stable `id` (derived from the source dataset's own row
    id, so baseline and later RAG runs can be compared row by row)."""
    slake = (fetch_slake_chest_records(split="test")
             + fetch_slake_chest_records(split="validation"))
    vqa_rad = fetch_vqa_rad_chest_records(splits=("test", "train"))
    all_records = slake + vqa_rad
    if verbose:
        print("Available pool before sampling:", pool_summary(all_records))

    sampled = stratified_sample(all_records, n_per_category=n_per_category, seed=seed,
                                max_per_question=max_per_question)

    slake_records = [r for r in sampled if r["source_dataset"] == "SLAKE"]
    vqa_rad_records = [r for r in sampled if r["source_dataset"] == "VQA-RAD"]
    slake_paths = download_slake_images(slake_records, images_dir)
    vqa_rad_paths = download_vqa_rad_images(vqa_rad_records, images_dir)

    for r in sampled:
        paths = slake_paths if r["source_dataset"] == "SLAKE" else vqa_rad_paths
        r["image_path"] = paths[r["image_key"]]
        r["id"] = r["uid"]

    return sampled


In [ ]:
"""Answer scoring for the baseline run.

Med-Flamingo's own finding (clinician ratings disagreeing sharply with
automated text-overlap metrics) is why several automated scores are reported
side by side rather than one. None is a substitute for a human spot-check.

- `exact_match`: strict, normalized string equality. Fair for yes/no, harsh for
  open-ended answers.
- `loose_match`: substring containment either direction after normalization.
- `token_f1`: word-overlap F1 (SQuAD style). Gives partial credit to open
  answers like "Chest tightness, shortness of breath" vs "Chest tightness,
  fatigue", which exact_match scores 0.

Every accuracy is reported with a Wilson 95% confidence interval, because with
a few dozen questions per bucket most differences between buckets are not
statistically meaningful, and the report should say so.

Yes/no questions get extra treatment (sensitivity/specificity and the
"always answer the majority class" baseline): a model that answers "no" to
almost everything scores well when most true answers are "no", but misses
real findings.
"""


import math
import re
from dataclasses import dataclass, field

_YES_SYNONYMS = {"yes", "y", "yeah", "affirmative", "true", "present"}
_NO_SYNONYMS = {"no", "n", "nope", "negative", "false", "absent", "not present"}

_PUNCT_RE = re.compile(r"[^a-z0-9\s]")
_WS_RE = re.compile(r"\s+")


def normalize(text: str) -> str:
    text = text.lower().strip()
    text = _PUNCT_RE.sub(" ", text)
    text = _WS_RE.sub(" ", text).strip()
    if text in _YES_SYNONYMS:
        return "yes"
    if text in _NO_SYNONYMS:
        return "no"
    return text


def exact_match(prediction: str, gold: str) -> bool:
    return normalize(prediction) == normalize(gold)


def loose_match(prediction: str, gold: str) -> bool:
    p, g = normalize(prediction), normalize(gold)
    if not p or not g:
        return False
    if p == g:
        return True
    # Guard against trivially short strings matching almost anything.
    shorter, longer = (p, g) if len(p) <= len(g) else (g, p)
    if len(shorter) < 3:
        return False
    return shorter in longer


def token_f1(prediction: str, gold: str) -> float:
    p_tokens, g_tokens = normalize(prediction).split(), normalize(gold).split()
    if not p_tokens or not g_tokens:
        return float(p_tokens == g_tokens)
    remaining = list(g_tokens)
    common = 0
    for tok in p_tokens:
        if tok in remaining:
            remaining.remove(tok)
            common += 1
    if common == 0:
        return 0.0
    precision, recall = common / len(p_tokens), common / len(g_tokens)
    return 2 * precision * recall / (precision + recall)


def wilson_interval(correct: int, total: int, z: float = 1.96) -> tuple[float, float]:
    if total == 0:
        return (float("nan"), float("nan"))
    p = correct / total
    denom = 1 + z * z / total
    center = (p + z * z / (2 * total)) / denom
    half = z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


@dataclass
class Stats:
    total: int = 0
    exact_correct: int = 0
    loose_correct: int = 0
    f1_sum: float = 0.0

    def add(self, em: bool, lm: bool, f1: float) -> None:
        self.total += 1
        self.exact_correct += int(em)
        self.loose_correct += int(lm)
        self.f1_sum += f1

    @property
    def exact_accuracy(self) -> float:
        return self.exact_correct / self.total if self.total else float("nan")

    @property
    def loose_accuracy(self) -> float:
        return self.loose_correct / self.total if self.total else float("nan")

    @property
    def mean_f1(self) -> float:
        return self.f1_sum / self.total if self.total else float("nan")

    @property
    def exact_ci(self) -> tuple[float, float]:
        return wilson_interval(self.exact_correct, self.total)


# Backwards-compatible name used by earlier notebook versions and tests.
CategoryStats = Stats


@dataclass
class YesNoStats:
    """Confusion counts for yes/no questions, treating 'yes' as the positive
    class (i.e. 'there is a finding')."""
    tp: int = 0   # gold yes, predicted yes
    fn: int = 0   # gold yes, predicted not-yes  (missed finding)
    tn: int = 0   # gold no,  predicted no
    fp: int = 0   # gold no,  predicted not-no   (false alarm)

    @property
    def total(self) -> int:
        return self.tp + self.fn + self.tn + self.fp

    @property
    def sensitivity(self) -> float:
        pos = self.tp + self.fn
        return self.tp / pos if pos else float("nan")

    @property
    def specificity(self) -> float:
        neg = self.tn + self.fp
        return self.tn / neg if neg else float("nan")

    @property
    def accuracy(self) -> float:
        return (self.tp + self.tn) / self.total if self.total else float("nan")

    @property
    def majority_baseline(self) -> float:
        """Accuracy of always answering the more common gold class."""
        return max(self.tp + self.fn, self.tn + self.fp) / self.total if self.total else float("nan")

    @property
    def predicted_yes_rate(self) -> float:
        return (self.tp + self.fp) / self.total if self.total else float("nan")

    @property
    def gold_yes_rate(self) -> float:
        return (self.tp + self.fn) / self.total if self.total else float("nan")


@dataclass
class EvaluationReport:
    per_category: dict[str, Stats] = field(default_factory=dict)
    per_source: dict[str, Stats] = field(default_factory=dict)
    per_format: dict[str, Stats] = field(default_factory=dict)
    per_category_format: dict[str, Stats] = field(default_factory=dict)
    per_kind: dict[str, Stats] = field(default_factory=dict)
    overall: Stats = field(default_factory=Stats)
    yes_no: YesNoStats = field(default_factory=YesNoStats)
    yes_no_by_category: dict[str, YesNoStats] = field(default_factory=dict)
    rows: list[dict] = field(default_factory=list)


def _gold_format(rec: dict) -> str:
    if rec.get("answer_format"):
        return rec["answer_format"]
    return "yes/no" if normalize(rec["gold_answer"]) in ("yes", "no") else "open"


def evaluate(records: list[dict]) -> EvaluationReport:
    """Score a list of records.

    Required keys: 'question', 'gold_answer', 'prediction', 'category',
    'source_dataset'. Optional: 'answer_format' ('yes/no'|'open', inferred if
    absent) and 'is_knowledge' (bool). Returns overall and per-bucket stats plus
    per-row scoring so a human can inspect individual disagreements.
    """
    report = EvaluationReport()

    for rec in records:
        em = exact_match(rec["prediction"], rec["gold_answer"])
        lm = loose_match(rec["prediction"], rec["gold_answer"])
        f1 = token_f1(rec["prediction"], rec["gold_answer"])
        fmt = _gold_format(rec)
        kind = "knowledge" if rec.get("is_knowledge") else "perception"

        for bucket, key in (
            (report.per_category, rec["category"]),
            (report.per_source, rec["source_dataset"]),
            (report.per_format, fmt),
            (report.per_category_format, f"{rec['category']} / {fmt}"),
            (report.per_kind, kind),
        ):
            bucket.setdefault(key, Stats()).add(em, lm, f1)
        report.overall.add(em, lm, f1)

        if fmt == "yes/no":
            gold_yes = normalize(rec["gold_answer"]) == "yes"
            pred = normalize(rec["prediction"])
            for yn in (report.yes_no, report.yes_no_by_category.setdefault(rec["category"], YesNoStats())):
                if gold_yes:
                    yn.tp += int(pred == "yes")
                    yn.fn += int(pred != "yes")
                else:
                    yn.tn += int(pred == "no")
                    yn.fp += int(pred != "no")

        report.rows.append({**rec, "answer_format": fmt, "exact_match": em,
                            "loose_match": lm, "token_f1": round(f1, 3)})

    return report


def _fmt_stats(s: Stats) -> str:
    lo, hi = s.exact_ci
    return (f"n={s.total:<4} exact={s.exact_accuracy:6.1%} (95% CI {lo:.0%}-{hi:.0%})"
            f"  loose={s.loose_accuracy:6.1%}  token_f1={s.mean_f1:.2f}")


def format_report(report: EvaluationReport) -> str:
    L = []
    L.append("=== Overall ===")
    L.append(_fmt_stats(report.overall))

    def section(title, bucket):
        L.append("")
        L.append(f"=== {title} ===")
        for key, s in sorted(bucket.items()):
            L.append(f"{key:>24}: {_fmt_stats(s)}")

    section("By question category (laterality / severity / general)", report.per_category)
    section("By answer format (yes/no vs open-ended)", report.per_format)
    section("By category and answer format", report.per_category_format)
    section("By question kind (knowledge = symptoms/treatment/etc; perception = look at the image)", report.per_kind)
    section("By source dataset", report.per_source)

    yn = report.yes_no
    L.append("")
    L.append("=== Yes/no questions: does the model miss findings? ===")
    if yn.total:
        L.append(f"n={yn.total}  accuracy={yn.accuracy:.1%}  "
                 f"always-answer-majority baseline={yn.majority_baseline:.1%}")
        L.append(f"sensitivity (correct on gold 'yes')={yn.sensitivity:.1%}  "
                 f"specificity (correct on gold 'no')={yn.specificity:.1%}")
        L.append(f"gold 'yes' rate={yn.gold_yes_rate:.1%}  model predicted 'yes' rate={yn.predicted_yes_rate:.1%}")
        for cat, c in sorted(report.yes_no_by_category.items()):
            L.append(f"  {cat:>12}: n={c.total:<3} acc={c.accuracy:.0%}  sens={c.sensitivity:.0%}  spec={c.specificity:.0%}")
    else:
        L.append("(no yes/no questions in this run)")
    L.append("")
    L.append("Note: buckets with n < ~50 have wide confidence intervals; treat differences that "
             "the intervals overlap on as unproven.")
    return "\n".join(L)


In [ ]:
"""Small, dependency-free statistical tests used to compare buckets and runs.

Everything here is implemented with the standard library so it runs in the CI
job and in Colab without scipy. Each function is checked against known values
in `tests/test_stats.py`.
"""


import math
import random


def binom_pmf(k: int, n: int, p: float) -> float:
    if p <= 0.0:
        return float(k == 0)
    if p >= 1.0:
        return float(k == n)
    return math.comb(n, k) * (p ** k) * ((1 - p) ** (n - k))


def binom_two_sided(k: int, n: int, p: float = 0.5) -> float:
    """Exact two-sided binomial test: probability, under Binomial(n, p), of an
    outcome no more likely than the observed `k`."""
    if n == 0:
        return 1.0
    observed = binom_pmf(k, n, p)
    total = sum(binom_pmf(i, n, p) for i in range(n + 1) if binom_pmf(i, n, p) <= observed * (1 + 1e-9))
    return min(1.0, total)


def mcnemar_exact(only_a_correct: int, only_b_correct: int) -> float:
    """Exact McNemar test on discordant pairs: under 'no difference between the
    two systems', each discordant pair is equally likely to favour either."""
    return binom_two_sided(only_a_correct, only_a_correct + only_b_correct, 0.5)


def fisher_exact_2x2(a: int, b: int, c: int, d: int) -> float:
    """Two-sided Fisher exact test for the table [[a, b], [c, d]]."""
    n1, n2, k = a + b, c + d, a + c
    n = n1 + n2

    def pmf(x: int) -> float:
        return math.comb(n1, x) * math.comb(n2, k - x) / math.comb(n, k)

    lo, hi = max(0, k - n2), min(n1, k)
    observed = pmf(a)
    return min(1.0, sum(pmf(x) for x in range(lo, hi + 1) if pmf(x) <= observed * (1 + 1e-9)))


def cluster_bootstrap_mean_diff(diffs: list[float], clusters: list[str], n_boot: int = 10000,
                                seed: int = 0, alpha: float = 0.05) -> tuple[float, float, float]:
    """Mean of paired differences with a percentile bootstrap CI that resamples
    whole clusters (e.g. all questions about one image), because questions about
    the same image or from the same template are not independent. Returns
    (mean, ci_low, ci_high)."""
    if not diffs:
        return (float("nan"), float("nan"), float("nan"))
    groups: dict[str, list[float]] = {}
    for d, c in zip(diffs, clusters):
        groups.setdefault(c, []).append(d)
    keys = list(groups)
    rng = random.Random(seed)
    means = []
    for _ in range(n_boot):
        total, count = 0.0, 0
        for _ in keys:
            g = groups[keys[rng.randrange(len(keys))]]
            total += sum(g)
            count += len(g)
        means.append(total / count)
    means.sort()
    lo = means[int((alpha / 2) * n_boot)]
    hi = means[min(n_boot - 1, int((1 - alpha / 2) * n_boot))]
    return (sum(diffs) / len(diffs), lo, hi)


In [ ]:
"""Scoring for 'which side' questions.

Gold answers to side questions range from "Right" to "Lower Left Lung", and a
model that answers "left lung" to "Lower Left Lung" has the side right even
though exact match scores it 0. So side questions are scored on the extracted
side (left / right / both), in addition to the usual metrics.

The same extraction supports the specific follow-up question from run 2: when
the correct side is left or both, does the model still answer "right"?
"""


import re


_BOTH_RE = re.compile(r"\b(bilateral|bilaterally|both)\b")
_LEFT_RE = re.compile(r"\bleft\b")
_RIGHT_RE = re.compile(r"\bright\b")


def side_of(text: str) -> str | None:
    """'left', 'right' or 'both' if the text names a side, else None.

    Text naming both words ("left and right lungs") counts as 'both'. Answers
    such as "No" or "Lung" name no side and return None."""
    t = str(text).lower()
    if _BOTH_RE.search(t):
        return "both"
    has_l, has_r = bool(_LEFT_RE.search(t)), bool(_RIGHT_RE.search(t))
    if has_l and has_r:
        return "both"
    if has_l:
        return "left"
    if has_r:
        return "right"
    return None


def score_side_rows(rows: list[dict]) -> list[dict]:
    """Add gold_side / pred_side / side_correct / side_given to each row."""
    out = []
    for r in rows:
        gold, pred = side_of(r["gold_answer"]), side_of(r["prediction"])
        out.append({**r, "gold_side": gold, "pred_side": pred,
                    "side_given": pred is not None, "side_correct": pred == gold})
    return out


def side_report(rows: list[dict]) -> dict:
    """Confusion table and bias statistics for rows scored by `score_side_rows`
    (rows whose gold names no side are ignored)."""
    rows = [r for r in rows if r.get("gold_side")]
    sides = ("left", "right", "both")
    confusion = {g: {p: 0 for p in (*sides, "none")} for g in sides}
    for r in rows:
        confusion[r["gold_side"]][r["pred_side"] or "none"] += 1

    per_gold = {}
    for g in sides:
        n = sum(confusion[g].values())
        per_gold[g] = {"n": n, "correct": confusion[g][g],
                       "accuracy": confusion[g][g] / n if n else float("nan")}

    # Asymmetry: among gold-left items that got a left/right answer, how often
    # was it "right"; among gold-right items, how often "left"? Equal rates
    # mean errors are symmetric; a higher left->right rate is a right default.
    l_to_r, l_to_l = confusion["left"]["right"], confusion["left"]["left"]
    r_to_l, r_to_r = confusion["right"]["left"], confusion["right"]["right"]
    asym_p = (fisher_exact_2x2(l_to_r, l_to_l, r_to_l, r_to_r)
              if (l_to_r + l_to_l) and (r_to_l + r_to_r) else float("nan"))

    # Among left/right-gold items answered left or right, is accuracy above 50%?
    lr = [r for r in rows if r["gold_side"] in ("left", "right") and r["pred_side"] in ("left", "right")]
    lr_correct = sum(r["side_correct"] for r in lr)

    pred_right = sum(1 for r in rows if r["pred_side"] == "right")
    gold_right = sum(1 for r in rows if r["gold_side"] == "right")
    return {
        "n": len(rows),
        "confusion": confusion,
        "per_gold": per_gold,
        "left_to_right_rate": l_to_r / (l_to_r + l_to_l) if (l_to_r + l_to_l) else float("nan"),
        "right_to_left_rate": r_to_l / (r_to_l + r_to_r) if (r_to_l + r_to_r) else float("nan"),
        "asymmetry_p": asym_p,
        "lr_answered": len(lr),
        "lr_correct": lr_correct,
        "lr_vs_chance_p": binom_two_sided(lr_correct, len(lr), 0.5) if lr else float("nan"),
        "predicted_right_rate": pred_right / len(rows) if rows else float("nan"),
        "gold_right_rate": gold_right / len(rows) if rows else float("nan"),
        "no_side_answer": sum(1 for r in rows if r["pred_side"] is None),
    }


def format_side_report(rep: dict) -> str:
    L = [f"=== Side questions (n={rep['n']}) ===", "gold \\ predicted      left  right  both  none  accuracy"]
    for g in ("left", "right", "both"):
        c, pg = rep["confusion"][g], rep["per_gold"][g]
        L.append(f"{g:>20}  {c['left']:5d}  {c['right']:5d}  {c['both']:4d}  {c['none']:4d}"
                 f"  {pg['correct']}/{pg['n']} ({pg['accuracy']:.0%})" if pg["n"] else f"{g:>20}  (none)")
    L.append("")
    L.append(f"model answered 'right' on {rep['predicted_right_rate']:.0%} of questions; "
             f"'right' was correct on {rep['gold_right_rate']:.0%}")
    L.append(f"gold-left answered 'right': {rep['left_to_right_rate']:.0%}   "
             f"gold-right answered 'left': {rep['right_to_left_rate']:.0%}   "
             f"(Fisher p for asymmetry = {rep['asymmetry_p']:.3f})")
    L.append(f"left-vs-right accuracy among items answered left/right: "
             f"{rep['lr_correct']}/{rep['lr_answered']}   (exact binomial p vs 50% chance = {rep['lr_vs_chance_p']:.3f})")
    L.append(f"questions where the model named no side: {rep['no_side_answer']}")
    return "\n".join(L)


In [ ]:
"""Two supplementary test sets built from the same public sources as the 224-question baseline.

1. Knowledge set: every symptoms / treatment / prevention / cause / function /
   'why' question available in SLAKE (all splits) and VQA-RAD. Run 2 had only 24,
   too few to detect a retrieval improvement. The pool is small (about 130) and
   made of a few fixed templates, so analysis must treat questions about the
   same image or template as correlated (see compare.py).

2. Side set: open questions whose gold answer names a side ('Right', 'Lower Left
   Lung', 'Both Lung'), sampled with equal numbers of left and right answers so
   a model that favours one side cannot score well by habit. Run 2 hinted at a
   right-side default from only 13 such questions.

Both sets include SLAKE train rows. CheXagent was very likely trained on those
(CheXinstruct includes SLAKE), so every record carries `source_split` and
results should be reported for train and test/validation rows separately.
Train rows make the set larger, not cleaner.
"""


import hashlib
import json
import random
from pathlib import Path


KNOWLEDGE_PROMPT = "{question}"
SIDE_PROMPT = "{question} Answer concisely based only on what is visible in the image."


def prompt_for_set(set_name: str, question: str, category: str) -> str:
    """Prompt used when running a supplementary set.

    knowledge: the plain question for every row. Run 2 appended "based only on
    what is visible in the image" to laterality/severity-tagged rows, which
    included 20 of its 24 knowledge questions, a confound for a question that
    is answered from knowledge.
    side: the visible-in-image suffix for every row (what run 2 used for
    laterality questions).
    """
    if set_name == "knowledge":
        return KNOWLEDGE_PROMPT.format(question=question)
    if set_name == "side":
        return SIDE_PROMPT.format(question=question)
    raise ValueError(f"unknown set: {set_name}")


def fetch_full_pool(include_train: bool = True, cache_path: str | Path | None = None) -> list[dict]:
    """All chest candidates from SLAKE (test, validation, train) and VQA-RAD.
    Cached as JSON because the full fetch takes about a minute."""
    if cache_path and Path(cache_path).exists():
        return json.loads(Path(cache_path).read_text())
    splits = ["test", "validation"] + (["train"] if include_train else [])
    pool: list[dict] = []
    for split in splits:
        pool += fetch_slake_chest_records(split=split)
    pool += fetch_vqa_rad_chest_records(splits=("test", "train"))
    if cache_path:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        Path(cache_path).write_text(json.dumps(pool))
    return pool


def _image_identity(r: dict) -> str:
    """Stable identity of the image behind a question. VQA-RAD rows carry a
    content hash, so questions about one image are recognised as sharing it."""
    return f"{r['source_dataset']}:{r.get('image_id') or r['image_key'].split('?')[0]}"


def select_knowledge_records(pool: list[dict]) -> list[dict]:
    """Knowledge questions, deduplicated on (source, image, question text)."""
    seen: set[tuple[str, str, str]] = set()
    out = []
    for r in pool:
        if not r["is_knowledge"]:
            continue
        key = (r["source_dataset"], _image_identity(r), r["question"].lower().strip())
        if key in seen:
            continue
        seen.add(key)
        out.append({**r, "knowledge_subtype": knowledge_subtype(r["question"])})
    return out


def select_side_records(pool: list[dict], n_per_side: int = 60, n_both: int = 20,
                        max_per_question_per_side: int = 8, seed: int = 0) -> list[dict]:
    """Open questions whose short gold answer names a side, balanced left/right.

    `max_per_question_per_side` stops one phrasing ("Where is the abnormality
    in this image?") from dominating a side.
    """
    candidates: dict[str, list[dict]] = {"left": [], "right": [], "both": []}
    for r in pool:
        if r["answer_format"] != "open" or r["is_knowledge"]:
            continue
        gold = r["gold_answer"]
        side = side_of(gold)
        if side is None or len(gold.split()) > 4:
            continue
        candidates[side].append({**r, "gold_side": side,
                                 "side_granularity": "side_only" if len(gold.split()) == 1 else "region"})

    rng = random.Random(seed)
    chosen: list[dict] = []
    for side, limit in (("left", n_per_side), ("right", n_per_side), ("both", n_both)):
        rows = candidates[side][:]
        rng.shuffle(rows)
        per_q: dict[str, int] = {}
        taken = []
        for r in rows:
            k = r["question"].lower().strip()
            if per_q.get(k, 0) >= max_per_question_per_side:
                continue
            per_q[k] = per_q.get(k, 0) + 1
            taken.append(r)
            if len(taken) >= limit:
                break
        chosen += taken
    rng.shuffle(chosen)
    return chosen


def split_dev_heldout(records: list[dict], dev_fraction: float = 0.3) -> tuple[list[dict], list[dict]]:
    """Deterministic split by image so no image appears on both sides.

    Retrieval design (what to retrieve, how many passages, prompt wording) may
    be tuned on the dev part only; the held-out part is scored once, at the end.
    Without this, a small test set gets tuned until it shows an improvement.
    """
    dev, held = [], []
    for r in records:
        bucket = int(hashlib.md5(_image_identity(r).encode()).hexdigest(), 16) % 1000
        (dev if bucket < dev_fraction * 1000 else held).append(r)
    return dev, held


def attach_images(records: list[dict], images_dir: str | Path) -> list[dict]:
    slake = download_slake_images([r for r in records if r["source_dataset"] == "SLAKE"], images_dir)
    vqa = download_vqa_rad_images([r for r in records if r["source_dataset"] == "VQA-RAD"], images_dir)
    for r in records:
        paths = slake if r["source_dataset"] == "SLAKE" else vqa
        r["image_path"] = paths[r["image_key"]]
        r["id"] = r["uid"]
    return records


def describe_set(records: list[dict]) -> dict:
    """Counts used in the documentation of how each set is composed."""
    def count(key):
        out: dict[str, int] = {}
        for r in records:
            out[str(r.get(key))] = out.get(str(r.get(key)), 0) + 1
        return dict(sorted(out.items()))

    return {
        "n": len(records),
        "by_source": count("source_dataset"),
        "by_split": count("source_split"),
        "by_subtype": count("knowledge_subtype") if records and "knowledge_subtype" in records[0] else None,
        "by_gold_side": count("gold_side") if records and "gold_side" in records[0] else None,
        "unique_questions": len({r["question"].lower().strip() for r in records}),
        "unique_gold_answers": len({r["gold_answer"].lower().strip() for r in records}),
        "unique_images": len({_image_identity(r) for r in records}),
    }


def build_knowledge_test_set(images_dir: str | Path, include_train: bool = True,
                             cache_path: str | Path | None = None, pool: list[dict] | None = None) -> list[dict]:
    """Pass `pool` (from `fetch_full_pool`) to build several sets from one fetch."""
    pool = pool if pool is not None else fetch_full_pool(include_train=include_train, cache_path=cache_path)
    return attach_images(select_knowledge_records(pool), images_dir)


def build_side_test_set(images_dir: str | Path, n_per_side: int = 60, n_both: int = 20, seed: int = 0,
                        include_train: bool = True, cache_path: str | Path | None = None,
                        pool: list[dict] | None = None) -> list[dict]:
    pool = pool if pool is not None else fetch_full_pool(include_train=include_train, cache_path=cache_path)
    return attach_images(select_side_records(pool, n_per_side=n_per_side, n_both=n_both, seed=seed), images_dir)


def main() -> None:
    import argparse

    parser = argparse.ArgumentParser(description="Build the knowledge and side test sets (no GPU needed).")
    parser.add_argument("--out", default="data")
    parser.add_argument("--images-dir", default="data/images")
    parser.add_argument("--skip-images", action="store_true", help="metadata only, no image download")
    args = parser.parse_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    pool = fetch_full_pool(cache_path=out / "cache" / "pools_full.json")
    sets = {"knowledge": select_knowledge_records(pool), "side": select_side_records(pool)}
    summary = {}
    for name, records in sets.items():
        if not args.skip_images:
            attach_images(records, args.images_dir)
        else:
            for r in records:
                r["id"] = r["uid"]
        with open(out / f"{name}_test_set.jsonl", "w") as f:
            for r in records:
                f.write(json.dumps(r) + "\n")
        summary[name] = describe_set(records)
        print(name, json.dumps(summary[name], indent=2))
    (out / "extra_sets_summary.json").write_text(json.dumps(summary, indent=2))


In [ ]:
"""Cheap perception diagnostics for CheXagent-2-3b (no training, no retrieval).

Run 3 found two perception faults: a constant "Right" on explicit "which side"
questions, and "no" to every heart-enlargement question whose answer was "yes".
These tests ask what could explain them, using the same model:

A. Flip test. Mirror each image left-right. A model that reads laterality from
   the image should swap its left/right answers; a model giving a constant answer
   will not. (Mirroring also moves the heart to the right side of the picture, so
   anatomy-based cues flip as well; that is consistent with a swap, not against it.)
B. Prompt variants on side questions. Does wording, a forced choice, describing
   first, or stating the left/right convention change the answers, and do the
   answers then depend on the true side?
C. Heart-size variants. Does describing the heart first, or asking "normal or
   enlarged?" instead of a yes/no question, get the model to ever say "enlarged"?

`docs/phase1c_diagnostics.md` states, before any result exists, what each outcome
would mean.
"""


import random
import re


OPPOSITE = {"left": "right", "right": "left"}

# ---------------------------------------------------------------- side prompts

SIDE_PROMPTS = {
    "suffix": "{question} Answer concisely based only on what is visible in the image.",
    "plain": "{question}",
    "one_word": "{question} Answer with exactly one word: Left or Right.",
    "describe_first": ("Describe in one sentence where the abnormality or finding is in this chest X-ray, "
                       "then answer the question. {question}"),
    "patient_side": ("{question} Left and right mean the patient's own left and right "
                     "(the patient's right side appears on the left of the picture)."),
    "image_side": "{question} Left and right mean the left and right of the picture as you look at it.",
}
SIDE_MAX_TOKENS = {"describe_first": 160}

_EXPLICIT_SIDE_RE = re.compile(r"left or right|which side")


def is_explicit_side_question(question: str) -> bool:
    """'Which side ...' or '... left or right?' questions, where the answer is
    a choice of side (as opposed to 'where is the abnormality?')."""
    return bool(_EXPLICIT_SIDE_RE.search(question.lower()))


def select_side_diagnostic_records(side_records: list[dict], n_other: int = 31, seed: int = 0) -> list[dict]:
    """Left/right-gold side questions: every explicit one plus `n_other` random
    others. Bilateral gold answers are excluded (they have no mirror image)."""
    lr = [r for r in side_records if r.get("gold_side") in ("left", "right")]
    explicit = [r for r in lr if is_explicit_side_question(r["question"])]
    others = [r for r in lr if not is_explicit_side_question(r["question"])]
    rng = random.Random(seed)
    rng.shuffle(others)
    return explicit + others[:n_other]


# --------------------------------------------------------------------- flip test

def flip_image(src_path: str, dest_path: str) -> str:
    """Write a left-right mirrored copy of an image."""
    from PIL import Image, ImageOps
    with Image.open(src_path) as im:
        ImageOps.mirror(im.convert("RGB")).save(dest_path)
    return dest_path


def flip_report(rows: list[dict]) -> dict:
    """Rows need: gold_side (left/right), pred_original, pred_flipped (model texts)
    and optionally `explicit` (bool). Reports how answers move when the image is mirrored."""
    def summarize(rs: list[dict]) -> dict:
        po = [side_of(r["pred_original"]) for r in rs]
        pf = [side_of(r["pred_flipped"]) for r in rs]
        both = [(a, b) for a, b in zip(po, pf) if a in OPPOSITE and b in OPPOSITE]
        swapped = sum(b == OPPOSITE[a] for a, b in both)
        same = sum(b == a for a, b in both)
        identical = sum(r["pred_original"].strip().lower() == r["pred_flipped"].strip().lower() for r in rs)
        acc_o = sum(a == r["gold_side"] for a, r in zip(po, rs))
        acc_f = sum(b == OPPOSITE[r["gold_side"]] for b, r in zip(pf, rs))
        return {
            "n": len(rs), "named_both": len(both), "swapped": swapped, "same": same,
            "swap_rate": swapped / len(both) if both else float("nan"),
            "swap_ci": wilson_interval(swapped, len(both)) if both else (float("nan"), float("nan")),
            "identical_text": identical,
            "acc_original": acc_o / len(rs) if rs else float("nan"),
            "acc_flipped_vs_flipped_gold": acc_f / len(rs) if rs else float("nan"),
            "right_rate_original": sum(a == "right" for a in po) / len(rs) if rs else float("nan"),
            "right_rate_flipped": sum(b == "right" for b in pf) / len(rs) if rs else float("nan"),
        }
    out = {"all": summarize(rows)}
    explicit = [r for r in rows if r.get("explicit")]
    if explicit:
        out["explicit"] = summarize(explicit)
    return out


def format_flip_report(rep: dict) -> str:
    L = ["=== Flip test: do left/right answers swap when the image is mirrored? ==="]
    for name, s in rep.items():
        lo, hi = s["swap_ci"]
        L.append(f"[{name}] n={s['n']}  answered left/right on both images: {s['named_both']}")
        L.append(f"   swapped: {s['swapped']}  stayed the same: {s['same']}   swap rate = {s['swap_rate']:.0%} "
                 f"(95% CI {lo:.0%}-{hi:.0%})")
        L.append(f"   byte-identical answers on both images: {s['identical_text']} / {s['n']}")
        L.append(f"   share answering 'right': original {s['right_rate_original']:.0%}, flipped {s['right_rate_flipped']:.0%}")
        L.append(f"   correct on original: {s['acc_original']:.0%}   correct on flipped (vs mirrored gold): "
                 f"{s['acc_flipped_vs_flipped_gold']:.0%}")
    return "\n".join(L)


# ------------------------------------------------------------ prompt variants (B)

def variant_report(rows: list[dict]) -> dict:
    """Rows need: gold_side (left/right) and prediction. `depends_on_truth_p` is the
    Fisher p for 'is the model's left/right answer related to the true side?'"""
    preds = [side_of(r["prediction"]) for r in rows]
    gl = [(p, r) for p, r in zip(preds, rows) if r["gold_side"] == "left"]
    gr = [(p, r) for p, r in zip(preds, rows) if r["gold_side"] == "right"]
    l_right = sum(p == "right" for p, _ in gl)
    l_left = sum(p == "left" for p, _ in gl)
    r_right = sum(p == "right" for p, _ in gr)
    r_left = sum(p == "left" for p, _ in gr)
    return {
        "n": len(rows),
        "named_side": sum(p in OPPOSITE for p in preds) / len(rows) if rows else float("nan"),
        "answered_right": sum(p == "right" for p in preds) / len(rows) if rows else float("nan"),
        "acc_gold_left": l_left / len(gl) if gl else float("nan"),
        "acc_gold_right": r_right / len(gr) if gr else float("nan"),
        "acc_overall": (l_left + r_right) / len(rows) if rows else float("nan"),
        "depends_on_truth_p": (fisher_exact_2x2(l_right, l_left, r_right, r_left)
                               if (l_right + l_left) and (r_right + r_left) else float("nan")),
    }


def format_variant_table(by_variant: dict[str, dict]) -> str:
    L = ["=== Side prompt variants ===",
         f"{'variant':>15} {'n':>4} {'named side':>10} {'said right':>10} {'acc gold-L':>10} {'acc gold-R':>10} "
         f"{'acc all':>8} {'depends on truth p':>19}"]
    for name, v in by_variant.items():
        L.append(f"{name:>15} {v['n']:>4} {v['named_side']:>10.0%} {v['answered_right']:>10.0%} "
                 f"{v['acc_gold_left']:>10.0%} {v['acc_gold_right']:>10.0%} {v['acc_overall']:>8.0%} "
                 f"{v['depends_on_truth_p']:>19.3f}")
    L.append("'depends on truth p' small (< 0.05) means the answer follows the true side; "
             "near 1 means it does not.")
    return "\n".join(L)


# ------------------------------------------------------------------ heart size (C)

_ENLARGE_POS = re.compile(r"enlarg|cardiomegaly|abnormally large|too big|too large|larger than normal|heart.*\blarge\b")
_HEART = re.compile(r"heart|cardiac|cardiomegaly")
_REVERSED = re.compile(r"normal|within|small|stretched|decreas|reduc")
_NOT_HEART = re.compile(r"pulmonary|hilar|bronch|vascul|aort|atri|ventric|pericard|mediastin")
_ON_LUNG = re.compile(r"\bon the (upper|lower|left|right|middle)[^?]*\blung")


def select_heart_size_records(pool: list[dict]) -> list[dict]:
    """Yes/no questions where 'yes' means the heart is enlarged: 'Is the heart
    enlarged?', 'Is there cardiomegaly?', 'Can cardiomegaly be observed in this
    picture?'. Excludes reversed wording ('Is the size normal?'), other structures
    (hilum, vessels, atria), and location-specific cardiomegaly questions, whose
    'no' answers are about the location."""
    out, seen = [], set()
    for r in pool:
        q = r["question"].lower()
        if r["answer_format"] != "yes/no" or not _HEART.search(q) or not _ENLARGE_POS.search(q):
            continue
        if _REVERSED.search(q) or _NOT_HEART.search(q) or _ON_LUNG.search(q):
            continue
        key = (r["source_dataset"], r["uid"])
        if key in seen:
            continue
        seen.add(key)
        out.append({**r, "gold_yes": r["gold_answer"].strip().lower().rstrip(".") == "yes"})
    return out


HEART_PROMPTS = {
    "run2_suffix": "{question} Answer concisely based only on what is visible in the image.",
    "plain": "{question}",
    "describe_first": ("{question} First describe the size of the heart compared with the width of the chest, "
                       "then finish with 'Answer: yes' or 'Answer: no'."),
    "normal_or_enlarged": "Is the heart size normal or enlarged? Answer with one word: normal or enlarged.",
}
HEART_MAX_TOKENS = {"describe_first": 160}

_ANSWER_RE = re.compile(r"answer\s*:\s*(yes|no)\b")
_YESNO_RE = re.compile(r"\b(yes|no)\b")
_NEGATED_ENLARGED = re.compile(r"\b(not|no|without|nor)\b[^.;]{0,25}\b(enlarged|enlargement|cardiomegaly|large)\b")


def parse_yes_no(text: str) -> str | None:
    """Final yes/no in a free-text answer: an 'Answer: yes/no' line if present,
    otherwise the last standalone yes or no."""
    t = text.lower()
    m = _ANSWER_RE.findall(t)
    if m:
        return m[-1]
    m = _YESNO_RE.findall(t)
    return m[-1] if m else None


def parse_normal_or_enlarged(text: str) -> str | None:
    """Map a 'normal / enlarged' answer to 'yes' (enlarged) or 'no' (not enlarged)."""
    t = text.lower()
    if _NEGATED_ENLARGED.search(t):
        return "no"
    if re.search(r"enlarg|cardiomegaly|\blarge\b|\bbig\b|increased", t):
        return "yes"
    if re.search(r"\bnormal\b|\bunremarkable\b", t):
        return "no"
    return None


def parse_heart_answer(variant: str, text: str) -> str | None:
    return parse_normal_or_enlarged(text) if variant == "normal_or_enlarged" else parse_yes_no(text)


def heart_report(rows: list[dict], variant: str) -> dict:
    """Rows need: gold_yes (bool) and prediction (model text)."""
    tp = fn = tn = fp = unparsed = 0
    for r in rows:
        p = parse_heart_answer(variant, r["prediction"])
        if p is None:
            unparsed += 1
        if r["gold_yes"]:
            tp += p == "yes"
            fn += p != "yes"
        else:
            tn += p == "no"
            fp += p != "no"
    pos, neg, n = tp + fn, tn + fp, len(rows)
    return {
        "n": n, "gold_yes": pos, "gold_no": neg, "tp": tp, "fn": fn, "tn": tn, "fp": fp, "unparsed": unparsed,
        "sensitivity": tp / pos if pos else float("nan"),
        "specificity": tn / neg if neg else float("nan"),
        "accuracy": (tp + tn) / n if n else float("nan"),
        "always_no_accuracy": neg / n if n else float("nan"),
        "predicted_yes_rate": (tp + fp) / n if n else float("nan"),
        # Youden's J = sensitivity + specificity - 1: 0 for a system that ignores the image (including one
        # that answers "yes" to everything), 1 for a perfect one.
        "youden_j": (tp / pos + tn / neg - 1) if pos and neg else float("nan"),
    }


def format_heart_table(by_variant: dict[str, dict]) -> str:
    L = ["=== Heart-size variants (positive = heart enlarged) ===",
         f"{'variant':>20} {'n':>3} {'yes in gold':>11} {'sensitivity':>12} {'specificity':>12} {'accuracy':>9} "
         f"{'always-no':>9} {'said yes':>9} {'Youden J':>9} {'unparsed':>8}"]
    for name, h in by_variant.items():
        sens = f"{h['tp']}/{h['gold_yes']} ({h['sensitivity']:.0%})"
        spec = f"{h['tn']}/{h['gold_no']} ({h['specificity']:.0%})"
        L.append(f"{name:>20} {h['n']:>3} {h['gold_yes']:>11} {sens:>12} {spec:>12} {h['accuracy']:>9.0%} "
                 f"{h['always_no_accuracy']:>9.0%} {h['predicted_yes_rate']:>9.0%} {h['youden_j']:>9.2f} {h['unparsed']:>8}")
    L.append("Youden J = sensitivity + specificity - 1. About 0 means no skill, even if the model says 'yes' more often.")
    return "\n".join(L)


In [ ]:
import json, os, random
from collections import Counter

QUICK = False   # True = tiny smoke test (a few items per test), then set back to False for the real run

pool = fetch_full_pool()
side_all = attach_images(select_side_records(pool), "images")
flip_items = [r for r in side_all if r["gold_side"] in ("left", "right")]          # 120
diag_items = select_side_diagnostic_records(side_all)                               # 60, a subset of flip_items
heart_items = attach_images(select_heart_size_records(pool), "images")              # 47

if QUICK:
    rng = random.Random(0)
    flip_items = rng.sample(flip_items, 10)
    diag_items = [r for r in diag_items if r["id"] in {x["id"] for x in flip_items}] or flip_items[:6]
    heart_items = rng.sample(heart_items, 8)

os.makedirs("images_flipped", exist_ok=True)
for r in flip_items:
    r["flipped_path"] = flip_image(r["image_path"], f"images_flipped/{os.path.basename(r['image_path'])}")
for r in flip_items + diag_items:
    r["explicit"] = is_explicit_side_question(r["question"])

print("QUICK mode:", QUICK)
print("Flip test items:", len(flip_items), "| explicit side questions among them:", sum(r["explicit"] for r in flip_items))
print("Side prompt-variant items:", len(diag_items), Counter(r["gold_side"] for r in diag_items))
print("Heart items:", len(heart_items), "| answer yes:", sum(r["gold_yes"] for r in heart_items),
      "| sources:", dict(Counter(r["source_dataset"] for r in heart_items)))


## 3. Load CheXagent-2-3b

Identical to the other notebooks (downloads about 12.6 GB the first time).

In [ ]:
import glob, re
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "StanfordAIMI/CheXagent-2-3b"
DEVICE = "cuda"
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("Using dtype:", DTYPE)

_VERSION_ASSERT = re.compile(r'^assert transformers\.__version__ == "4\.40\.0".*$', flags=re.M)

def patch_version_asserts(model_dir):
    """Neutralize the model code's hard-coded transformers==4.40.0 checks.
    Returns the number of lines patched (expected: 2)."""
    patched = 0
    for path in glob.glob(f"{model_dir}/*.py"):
        text = open(path).read()
        new_text, n = _VERSION_ASSERT.subn("pass", text)
        if n:
            open(path, "w").write(new_text)
            patched += n
    return patched

model_dir = snapshot_download(MODEL_NAME, local_dir="CheXagent-2-3b")
print("Patched version checks:", patch_version_asserts(model_dir))

tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_dir, device_map="auto", torch_dtype=DTYPE, trust_remote_code=True
)
model.eval()
print("Model loaded.")


In [ ]:
GENERAL_PROMPT = "{question}"
LATERALITY_SEVERITY_PROMPT = "{question} Answer concisely based only on what is visible in the image."

def prompt_for_category(question, category):
    """Deliberately minimal (no few-shot examples, no retrieval) -- that's the
    whole point of a *baseline*. Phase 2 is where retrieved context gets added."""
    if category in ("laterality", "severity"):
        return LATERALITY_SEVERITY_PROMPT.format(question=question)
    return GENERAL_PROMPT.format(question=question)


def ask_chexagent(image_path, question, max_new_tokens=128):
    query = tokenizer.from_list_format([{"image": image_path}, {"text": question}])
    conv = [
        {"from": "system", "value": "You are a helpful radiology assistant. Answer the question as concisely as possible based only on the image."},
        {"from": "human", "value": query},
    ]
    input_ids = tokenizer.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            input_ids.to(DEVICE), do_sample=False, num_beams=1, use_cache=True,
            max_new_tokens=max_new_tokens,
        )[0]
    return tokenizer.decode(output[input_ids.size(1):-1]).strip()


## 4. Run the tests

Every answer is stored under a key (`test|variant|question id`) and written to disk every 20 answers.

In [ ]:
from tqdm.auto import tqdm

CKPT = "diagnostics_checkpoint.json"
store = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
_since_save = 0

def ask_cached(key, image_path, prompt, max_new_tokens=128, force=False):
    global _since_save
    if key in store and not force:
        return store[key]
    store[key] = ask_chexagent(image_path, prompt, max_new_tokens)
    _since_save += 1
    if _since_save % 20 == 0:
        json.dump(store, open(CKPT, "w"))
    return store[key]

# --- Test A: original vs mirrored image, with the same prompt run 3 used for side questions
for r in tqdm(flip_items, desc="A: flip"):
    prompt = SIDE_PROMPTS["suffix"].format(question=r["question"])
    r["pred_original"] = ask_cached(f"A|original|{r['id']}", r["image_path"], prompt)
    r["pred_flipped"] = ask_cached(f"A|flipped|{r['id']}", r["flipped_path"], prompt)

# --- Repeatability check: the same question asked again should give the same answer (greedy decoding)
repeat_items = flip_items[:20]
same = 0
for r in tqdm(repeat_items, desc="A: repeat"):
    again = ask_cached(f"A|repeat|{r['id']}", r["image_path"],
                       SIDE_PROMPTS["suffix"].format(question=r["question"]), force=True)
    same += again == r["pred_original"]
print(f"Repeatability: {same} / {len(repeat_items)} identical answers on a second run")

# --- Test B: side prompt variants
by_id = {r["id"]: r for r in flip_items}
for variant, template in SIDE_PROMPTS.items():
    for r in tqdm(diag_items, desc=f"B: {variant}"):
        if variant == "suffix":
            r["pred_suffix"] = by_id[r["id"]]["pred_original"]
            continue
        r[f"pred_{variant}"] = ask_cached(f"B|{variant}|{r['id']}", r["image_path"],
                                          template.format(question=r["question"]),
                                          SIDE_MAX_TOKENS.get(variant, 128))

# --- Test C: heart-size variants
for variant, template in HEART_PROMPTS.items():
    for r in tqdm(heart_items, desc=f"C: {variant}"):
        r[f"pred_{variant}"] = ask_cached(f"C|{variant}|{r['id']}", r["image_path"],
                                          template.format(question=r["question"]),
                                          HEART_MAX_TOKENS.get(variant, 128))
json.dump(store, open(CKPT, "w"))
print("Done.")


## 5. Results

**A. Flip test.** `swap rate` is the share of items answered left/right on both images where the answer changed
side. A model that reads laterality should be near 100%. A constant answer gives near 0%. Mirroring also moves the heart
to the right of the picture, so anatomy-based cues flip too; that would show up as a swap, not hide one.


In [ ]:
print(format_flip_report(flip_report(flip_items)))
print(f"\nRepeatability check: {same} / {len(repeat_items)} identical on a second run "
      "(if this is well below 20, differences between runs are partly noise).")


**B. Side prompt variants** on the 60 side questions, and again on the explicit ones only.

In [ ]:
def rows_for(variant, items):
    return [{"gold_side": r["gold_side"], "prediction": r[f"pred_{variant}"]} for r in items]

print(format_variant_table({v: variant_report(rows_for(v, diag_items)) for v in SIDE_PROMPTS}))
explicit_items = [r for r in diag_items if r["explicit"]]
print(f"\nExplicit 'which side' questions only (n={len(explicit_items)}):")
print(format_variant_table({v: variant_report(rows_for(v, explicit_items)) for v in SIDE_PROMPTS}))


**C. Heart size.** Positive means "heart enlarged". Sensitivity is the share of enlarged hearts the model called
enlarged. `always-no` is the accuracy of answering "no" to everything. `unparsed` counts answers that contain no usable
yes/no or normal/enlarged (counted as wrong). The "describe first" variant is parsed from its last yes/no.


In [ ]:
def heart_rows(variant):
    return [{"gold_yes": r["gold_yes"], "prediction": r[f"pred_{variant}"]} for r in heart_items]

print(format_heart_table({v: heart_report(heart_rows(v), v) for v in HEART_PROMPTS}))
print("\nExamples of 'describe first' answers:")
for r in heart_items[:5]:
    print(f"  gold={'yes' if r['gold_yes'] else 'no'}: {r['pred_describe_first'][:200]!r}")


## 6. Save results

Keep these files. The Colab VM is wiped when the session ends.

In [ ]:
import csv

rows = []
for r in flip_items:
    for v, k in (("original", "pred_original"), ("flipped", "pred_flipped")):
        rows.append({"test": "A_flip", "variant": v, "id": r["id"], "source_dataset": r["source_dataset"],
                     "source_split": r["source_split"], "question": r["question"], "gold": r["gold_answer"],
                     "gold_side": r["gold_side"], "explicit": r["explicit"], "prediction": r[k]})
for v in SIDE_PROMPTS:
    for r in diag_items:
        rows.append({"test": "B_side_prompt", "variant": v, "id": r["id"], "source_dataset": r["source_dataset"],
                     "source_split": r["source_split"], "question": r["question"], "gold": r["gold_answer"],
                     "gold_side": r["gold_side"], "explicit": r["explicit"], "prediction": r[f"pred_{v}"]})
for v in HEART_PROMPTS:
    for r in heart_items:
        rows.append({"test": "C_heart", "variant": v, "id": r["id"], "source_dataset": r["source_dataset"],
                     "source_split": r["source_split"], "question": r["question"], "gold": r["gold_answer"],
                     "gold_side": "", "explicit": "", "prediction": r[f"pred_{v}"],
                     "parsed": parse_heart_answer(v, r[f"pred_{v}"])})

fields = ["test", "variant", "id", "source_dataset", "source_split", "question", "gold", "gold_side", "explicit",
          "prediction", "parsed"]
with open("diagnostics_results.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for row in rows:
        w.writerow({k: row.get(k, "") for k in fields})
print(f"Saved diagnostics_results.csv ({len(rows)} rows)")
try:
    from google.colab import files
    files.download("diagnostics_results.csv")
    files.download("diagnostics_checkpoint.json")
except Exception as exc:
    print("Auto-download unavailable; use the Colab file browser instead.", exc)
